# YgoMaster-PackBuilder — Rarity Mapping Tests

Test del rarity normalization layer con override per `PackType`.

Regole:
- `Common -> N`
- `Rare -> R`
- `Super Rare -> SR`
- `Ultra Rare -> UR`
- `Secret Rare -> N` nei main
- `Secret Rare -> N` nei mini
- `Secret Rare -> UR` negli shared staples


## 1. Repository root

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()
if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from tempfile import TemporaryDirectory

from ygomaster_packbuilder.domain import Card, PackType, Rarity
from ygomaster_packbuilder.mapping import (
    RarityMapping,
    load_rarity_mapping,
    normalize_card_rarity,
    normalize_cards,
    resolve_rarity,
)

print("Imports OK")

Imports OK


## 3. Caricamento mapping reale

In [3]:
mapping_path = repo_root / "config" / "rarity_mapping.yaml"
rarity_mapping = load_rarity_mapping(mapping_path)

assert isinstance(rarity_mapping, RarityMapping)
print(rarity_mapping)

RarityMapping(rarities={'Common': <Rarity.N: 'N'>, 'Rare': <Rarity.R: 'R'>, 'Super Rare': <Rarity.SR: 'SR'>, 'Ultra Rare': <Rarity.UR: 'UR'>}, overrides={<PackType.PERSONAL_MAIN: 'personal_main'>: {'Secret Rare': <Rarity.N: 'N'>}, <PackType.PERSONAL_MINI: 'personal_mini'>: {'Secret Rare': <Rarity.N: 'N'>}, <PackType.SHARED_STAPLES: 'shared_staples'>: {'Secret Rare': <Rarity.UR: 'UR'>}})


## 4. Mapping globale

In [4]:
assert rarity_mapping.rarities == {
    "Common": Rarity.N,
    "Rare": Rarity.R,
    "Super Rare": Rarity.SR,
    "Ultra Rare": Rarity.UR,
}

print("Global mapping OK")

Global mapping OK


## 5. Override Secret Rare

In [5]:
assert rarity_mapping.overrides[PackType.PERSONAL_MAIN]["Secret Rare"] is Rarity.N
assert rarity_mapping.overrides[PackType.PERSONAL_MINI]["Secret Rare"] is Rarity.N
assert rarity_mapping.overrides[PackType.SHARED_STAPLES]["Secret Rare"] is Rarity.UR

print("Overrides OK")

Overrides OK


## 6. Global rarities su tutti i PackType

In [6]:
for pack_type in PackType:
    assert resolve_rarity("Common", pack_type, rarity_mapping) is Rarity.N
    assert resolve_rarity("Rare", pack_type, rarity_mapping) is Rarity.R
    assert resolve_rarity("Super Rare", pack_type, rarity_mapping) is Rarity.SR
    assert resolve_rarity("Ultra Rare", pack_type, rarity_mapping) is Rarity.UR

print("Global rarities OK")

Global rarities OK


## 7. Secret Rare: main

In [7]:
assert resolve_rarity(
    "Secret Rare",
    PackType.PERSONAL_MAIN,
    rarity_mapping,
) is Rarity.N

print("PERSONAL_MAIN: Secret Rare -> N")

PERSONAL_MAIN: Secret Rare -> N


## 8. Secret Rare: mini

In [8]:
assert resolve_rarity(
    "Secret Rare",
    PackType.PERSONAL_MINI,
    rarity_mapping,
) is Rarity.N

print("PERSONAL_MINI: Secret Rare -> N")

PERSONAL_MINI: Secret Rare -> N


## 9. Secret Rare: shared staples

In [9]:
assert resolve_rarity(
    "Secret Rare",
    PackType.SHARED_STAPLES,
    rarity_mapping,
) is Rarity.UR

print("SHARED_STAPLES: Secret Rare -> UR")

SHARED_STAPLES: Secret Rare -> UR


## 10. Normalizzazione singola

In [10]:
card = Card(
    name="Secret Test",
    ydk_id=12345678,
    source_rarity="Secret Rare",
)

normalize_card_rarity(
    card,
    rarity_mapping,
    PackType.PERSONAL_MAIN,
)

assert card.rarity is Rarity.N
assert card.source_rarity == "Secret Rare"
assert card.master_duel_id is None

print(card)

Card(name='Secret Test', ydk_id=12345678, source_rarity='Secret Rare', master_duel_id=None, rarity=<Rarity.N: 'N'>)


## 11. Normalizzazione batch

In [11]:
cards = [
    Card(name="A", ydk_id=1, source_rarity="Common"),
    Card(name="B", ydk_id=2, source_rarity="Secret Rare"),
    Card(name="C", ydk_id=3, source_rarity="Ultra Rare"),
]

normalize_cards(
    cards,
    rarity_mapping,
    PackType.SHARED_STAPLES,
)

assert [card.rarity for card in cards] == [
    Rarity.N,
    Rarity.UR,
    Rarity.UR,
]

print(cards)

[Card(name='A', ydk_id=1, source_rarity='Common', master_duel_id=None, rarity=<Rarity.N: 'N'>), Card(name='B', ydk_id=2, source_rarity='Secret Rare', master_duel_id=None, rarity=<Rarity.UR: 'UR'>), Card(name='C', ydk_id=3, source_rarity='Ultra Rare', master_duel_id=None, rarity=<Rarity.UR: 'UR'>)]


## 12. Precedenza override

In [12]:
custom_mapping = RarityMapping(
    rarities={"Rare": Rarity.R},
    overrides={
        PackType.SHARED_STAPLES: {
            "Rare": Rarity.N,
        },
    },
)

assert resolve_rarity("Rare", PackType.PERSONAL_MAIN, custom_mapping) is Rarity.R
assert resolve_rarity("Rare", PackType.SHARED_STAPLES, custom_mapping) is Rarity.N

print("Override precedence OK")

Override precedence OK


## 13. Rarità sconosciuta

In [13]:
card = Card(
    name="Unknown",
    ydk_id=999,
    source_rarity="Collector's Rare",
)

try:
    normalize_card_rarity(
        card,
        rarity_mapping,
        PackType.PERSONAL_MAIN,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError("La rarità sconosciuta avrebbe dovuto essere rifiutata.")

assert card.rarity is None

Expected ValueError: Unsupported source rarity "Collector's Rare" for pack type <PackType.PERSONAL_MAIN: 'personal_main'>.


## 14. Lista vuota

In [14]:
cards = []
normalize_cards(cards, rarity_mapping, PackType.PERSONAL_MINI)
assert cards == []
print("Empty list OK")

Empty list OK


## 15. File inesistente

In [15]:
missing = repo_root / "config" / "__missing_rarity_mapping__.yaml"

try:
    load_rarity_mapping(missing)
except FileNotFoundError as exc:
    print("Expected FileNotFoundError:", exc)
else:
    raise AssertionError("Il file inesistente avrebbe dovuto fallire.")

Expected FileNotFoundError: Rarity mapping file not found: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\__missing_rarity_mapping__.yaml


## 16. Root YAML non mapping

In [16]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text("- Common\n- Rare\n", encoding="utf-8")
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Root non mapping non rifiutata.")

Expected ValueError: Rarity mapping root must be a mapping.


## 17. rarities mancante

In [17]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text(
        'overrides:\n  personal_main:\n    Secret Rare: "N"\n',
        encoding="utf-8",
    )
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("rarities mancante non rifiutata.")

Expected ValueError: Rarities must be a mapping.


## 18. Valore rarity globale non valido

In [18]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text('rarities:\n  Common: "INVALID"\n', encoding="utf-8")
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Rarity invalida non rifiutata.")

Expected ValueError: Invalid rarity value for key 'Common' in rarities: INVALID


## 19. PackType override sconosciuto

In [19]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text(
        'rarities:\n  Common: "N"\noverrides:\n  unknown_pack:\n    Secret Rare: "UR"\n',
        encoding="utf-8",
    )
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("PackType sconosciuto non rifiutato.")

Expected ValueError: Unknown pack type in rarity override: 'unknown_pack'


## 20. Override non mapping

In [20]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text(
        'rarities:\n  Common: "N"\noverrides:\n  personal_main: "N"\n',
        encoding="utf-8",
    )
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Override non mapping non rifiutato.")

Expected ValueError: Rarity override for ''personal_main'' must be a mapping.


## 21. Valore override non valido

In [21]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "mapping.yaml"
    path.write_text(
        'rarities:\n  Common: "N"\noverrides:\n  personal_main:\n    Secret Rare: "INVALID"\n',
        encoding="utf-8",
    )
    try:
        load_rarity_mapping(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError("Override invalido non rifiutato.")

Expected ValueError: Invalid rarity value for key 'Secret Rare' in overrides.personal_main: INVALID
